# 6단원 실습 — 모델 모니터링 및 유지보수
Google Colab / Evidently 0.7.23

2차시: Drift 시뮬레이션과 분포 변화 관찰  
3차시: Evidently Drift Report와 Monitoring Gate

In [ ]:
!pip -q install "evidently==0.7.23" "pandas<3" scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import evidently
from sklearn.datasets import load_iris

FEATURES=["sepal_length","sepal_width","petal_length","petal_width"]
print("pandas:",pd.__version__)
print("evidently:",evidently.__version__)

# 2차시 — Drift 시뮬레이션과 분포 변화 관찰

In [ ]:
iris=load_iris(as_frame=True)
df=iris.frame.copy().rename(columns={"sepal length (cm)":"sepal_length","sepal width (cm)":"sepal_width","petal length (cm)":"petal_length","petal width (cm)":"petal_width"})
df.head()

In [ ]:
rng=np.random.default_rng(42)
reference=df.copy()
current_normal=df.copy()
for col in FEATURES:
    current_normal[col]+=rng.normal(0,0.02,size=len(current_normal))

current_drift=df.copy()
current_drift["sepal_length"]+=1.5
current_drift["petal_length"]+=2.0
current_drift["petal_width"]*=1.6
for col in FEATURES:
    current_drift[col]+=rng.normal(0,0.03,size=len(current_drift))

print(reference.shape,current_normal.shape,current_drift.shape)

In [ ]:
pd.DataFrame({"Reference":reference[FEATURES].mean(),"Current Normal":current_normal[FEATURES].mean(),"Current Drift":current_drift[FEATURES].mean()}).round(3)

In [ ]:
pd.DataFrame({"Normal Shift":current_normal[FEATURES].mean()-reference[FEATURES].mean(),"Drift Shift":current_drift[FEATURES].mean()-reference[FEATURES].mean()}).round(3)

In [ ]:
def summary_stats(data):
    return data[FEATURES].describe().loc[["mean","std","25%","50%","75%"]].round(3)

display(summary_stats(reference))
display(summary_stats(current_normal))
display(summary_stats(current_drift))

In [ ]:
for feature in FEATURES:
    plt.figure(figsize=(7,4))
    plt.hist(reference[feature],bins=20,alpha=0.6,label="Reference")
    plt.hist(current_drift[feature],bins=20,alpha=0.6,label="Current Drift")
    plt.title(f"Reference vs Current Drift: {feature}")
    plt.xlabel(feature); plt.ylabel("Count"); plt.legend(); plt.show()

In [ ]:
for feature in FEATURES:
    plt.figure(figsize=(7,4))
    plt.hist(reference[feature],bins=20,alpha=0.6,label="Reference")
    plt.hist(current_normal[feature],bins=20,alpha=0.6,label="Current Normal")
    plt.title(f"Reference vs Current Normal: {feature}")
    plt.xlabel(feature); plt.ylabel("Count"); plt.legend(); plt.show()

In [ ]:
for feature in FEATURES:
    plt.figure(figsize=(6,4))
    plt.boxplot([reference[feature],current_drift[feature]],tick_labels=["Reference","Current Drift"])
    plt.title(f"Boxplot: {feature}"); plt.ylabel(feature); plt.show()

In [ ]:
rng_weak=np.random.default_rng(42)
current_weak_drift=df.copy()
current_weak_drift["sepal_length"]+=0.2
current_weak_drift["petal_length"]+=0.3
current_weak_drift["petal_width"]*=1.1
for col in FEATURES:
    current_weak_drift[col]+=rng_weak.normal(0,0.03,size=len(current_weak_drift))

pd.DataFrame({"Strong Drift":current_drift[FEATURES].mean()-reference[FEATURES].mean(),"Weak Drift":current_weak_drift[FEATURES].mean()-reference[FEATURES].mean()}).round(3)

In [ ]:
feature="petal_length"
plt.figure(figsize=(7,4))
plt.hist(reference[feature],bins=20,alpha=0.5,label="Reference")
plt.hist(current_weak_drift[feature],bins=20,alpha=0.5,label="Weak Drift")
plt.hist(current_drift[feature],bins=20,alpha=0.5,label="Strong Drift")
plt.title("Reference vs Weak / Strong Drift"); plt.xlabel(feature); plt.ylabel("Count"); plt.legend(); plt.show()

## 2차시 정리
Reference = 비교 기준, Current = 현재 운영 데이터, Data Drift = P(X)의 변화

# 3차시 — Evidently Drift Report와 Monitoring Gate

In [ ]:
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset

In [ ]:
definition=DataDefinition(numerical_columns=FEATURES)
reference_dataset=Dataset.from_pandas(reference[FEATURES],data_definition=definition)
normal_dataset=Dataset.from_pandas(current_normal[FEATURES],data_definition=definition)
drift_dataset=Dataset.from_pandas(current_drift[FEATURES],data_definition=definition)

In [ ]:
normal_report=Report([DataDriftPreset(columns=FEATURES,drift_share=0.5)])
normal_snapshot=normal_report.run(current_data=normal_dataset,reference_data=reference_dataset)
normal_snapshot

In [ ]:
drift_report=Report([DataDriftPreset(columns=FEATURES,drift_share=0.5)])
drift_snapshot=drift_report.run(current_data=drift_dataset,reference_data=reference_dataset)
drift_snapshot

In [ ]:
normal_snapshot.save_html("normal_drift_report.html")
drift_snapshot.save_html("drift_report.html")
print("HTML Report 저장 완료")

In [ ]:
drift_json=drift_snapshot.json()
print(drift_json[:2000])

In [ ]:
def get_drift_summary(snapshot):
    d=snapshot.dict()
    for metric in d.get("metrics",[]):
        if metric.get("metric_name","").startswith("DriftedColumnsCount"):
            value=metric.get("value",{})
            return {"count":value.get("count"),"share":value.get("share")}
    return {"count":None,"share":None}

normal_summary=get_drift_summary(normal_snapshot)
drift_summary=get_drift_summary(drift_snapshot)
print("Normal:",normal_summary)
print("Drift:",drift_summary)

### Drift Share
4개 Feature 중 3개가 Drift이면 Drift Share = 3/4 = 0.75.  
`drift_share`는 Dataset 수준의 정책 기준이며 개별 Feature 통계검정 threshold와 다릅니다.

In [ ]:
rows=[]
for threshold in [0.25,0.50,0.75]:
    report=Report([DataDriftPreset(columns=FEATURES,drift_share=threshold)])
    snapshot=report.run(current_data=drift_dataset,reference_data=reference_dataset)
    summary=get_drift_summary(snapshot)
    rows.append({"drift_share_threshold":threshold,"drifted_columns":summary["count"],"actual_drift_share":summary["share"]})
pd.DataFrame(rows)

In [ ]:
weak_drift_dataset=Dataset.from_pandas(current_weak_drift[FEATURES],data_definition=definition)
weak_report=Report([DataDriftPreset(columns=FEATURES,drift_share=0.5)])
weak_snapshot=weak_report.run(current_data=weak_drift_dataset,reference_data=reference_dataset)
weak_snapshot

In [ ]:
weak_summary=get_drift_summary(weak_snapshot)
weak_summary

In [ ]:
WARNING_THRESHOLD=0.5
current_share=drift_summary["share"]
print("Current Drift Share:",current_share)
if current_share is None:
    print("ERROR: Drift metric을 찾지 못함")
elif current_share>=WARNING_THRESHOLD:
    print("WARNING: Data Drift 감지")
    print("추가 분석 또는 Retraining 검토 필요")
else:
    print("PASS: Drift 수준이 허용 범위")

In [ ]:
normal_share=normal_summary["share"]
print("Normal Drift Share:",normal_share)
if normal_share is not None and normal_share>=WARNING_THRESHOLD:
    print("WARNING: Drift 감지")
else:
    print("PASS: Drift 수준이 허용 범위")

## 3차시 정리
Reference vs Current → Evidently → Drift Report → Drift Share → Monitoring Gate

Drift 감지 ≠ 모델 성능 저하 확정  
Drift 감지 ≠ 즉시 재학습  
Drift는 추가 분석이 필요한 Monitoring Signal입니다.